## 🎯 Learning Objectives
* Design and implement a REST API backend using FastAPI.
* Integrate LangChain components (LLMs, retrievers, prompt templates) into a FastAPI application.
* Utilize Pydantic for robust request and response data validation.
* Implement asynchronous API endpoints for improved performance and scalability.
* Understand the structure for deploying a FastAPI application, including dependency management.


## RAG03-L05: Building a REST API backend with FastAPI and LangChain

**Track**: Agentic AI & Automation Tools
**Course**: RAG-03 — Building End-to-End Generative AI Applications
**Section**: Building the Application

### Exercise Task: Develop a FastAPI Backend for a RAG Application

In this exercise, you will build a robust REST API backend using FastAPI that serves a Retrieval-Augmented Generation (RAG) system powered by LangChain. This backend will expose a single endpoint to accept user queries, process them through a RAG chain, and return the generated answer.

#### Requirements:

1.  **FastAPI Application**: Create a FastAPI application instance.
2.  **Pydantic Models**: Define Pydantic models for:
    *   `QueryRequest`: To validate incoming user queries (e.g., a `query` string).
    *   `QueryResponse`: To structure the outgoing response (e.g., an `answer` string).
3.  **LangChain Integration**: Implement a RAG chain using LangChain Expression Language (LCEL) that includes:
    *   A retriever (we will use a mock `FAISS` vector store for this exercise).
    *   A `ChatPromptTemplate` for structuring the RAG prompt.
    *   A Large Language Model (LLM) (e.g., `ChatOpenAI`).
    *   An `StrOutputParser` to extract the final answer.
4.  **API Endpoint**: Create an asynchronous `POST` endpoint at `/query` that:
    *   Accepts a `QueryRequest` object in the request body.
    *   Invokes the LangChain RAG chain with the user's query.
    *   Returns a `QueryResponse` object containing the generated answer.
5.  **Error Handling**: Implement basic error handling for unexpected issues during the RAG process.
6.  **Dependency Management**: Ensure your solution implicitly defines the necessary dependencies for a `requirements.txt` file.

#### Evaluation Criteria:

*   Correct implementation of FastAPI application and endpoint.
*   Proper use of Pydantic for request/response validation.
*   Successful integration and execution of a LangChain RAG chain.
*   Asynchronous endpoint definition and usage.
*   Clear and well-structured code with appropriate comments.
*   The API should be runnable locally using `uvicorn`.

Let's get started by setting up our environment and mock RAG components.


In [ ]:
# Install necessary libraries (run this cell once)
!pip install -q fastapi uvicorn langchain langchain-openai pydantic faiss-cpu python-dotenv

import os
from dotenv import load_dotenv
from typing import List

# --- Environment Setup ---
# Load environment variables from .env file (if present)
load_dotenv()

# Set your OpenAI API key. In a production environment, use a secure method like Kubernetes secrets or AWS Secrets Manager.
# For local development, ensure OPENAI_API_KEY is set in your .env file or as an environment variable.
# os.environ["OPENAI_API_KEY"] = "YOUR_OPENAI_API_KEY"

if not os.getenv("OPENAI_API_KEY"):
    print("WARNING: OPENAI_API_KEY environment variable not set. LangChain components requiring it may fail.")
    print("Please set it in your .env file or as an environment variable.")

# --- Mock RAG Component Setup ---
# In a real application, these would be loaded from persistent storage or external services.

from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

print("Setting up mock RAG components...")

# 1. Mock Documents and Embeddings
# For a real application, you'd load documents from a database, files, etc.
# and generate embeddings for them.
texts = [
    "The capital of France is Paris.",
    "The Eiffel Tower is located in Paris.",
    "Generative AI models can create text, images, and other media.",
    "FastAPI is a modern, fast (high-performance) web framework for building APIs with Python 3.8+ based on standard Python type hints.",
    "LangChain is a framework for developing applications powered by language models.",
    "RAG stands for Retrieval-Augmented Generation, a technique that enhances LLMs by providing external knowledge.",
    "Agentic AI systems combine LLMs with tools and planning capabilities to achieve complex goals."
]

# Initialize embeddings model
# Ensure OPENAI_API_KEY is set for this to work.
embeddings_model = OpenAIEmbeddings(model="text-embedding-3-small")

# Create a mock FAISS vector store from the texts
# In a real scenario, this would be loaded from disk or a cloud service.
vectorstore = FAISS.from_texts(texts, embeddings_model)
retriever = vectorstore.as_retriever()

print(f"Mock vector store created with {len(texts)} documents.")

# 2. Initialize LLM
# Using a modern OpenAI model. Adjust as needed.
llm = ChatOpenAI(model_name="gpt-4o-mini", temperature=0.1)

# 3. Define the RAG Prompt Template
# This template structures how the retrieved context and user query are presented to the LLM.
rag_prompt_template = ChatPromptTemplate.from_messages([
    ("system", "You are an AI assistant for question-answering tasks. Use the following retrieved context to answer the question. If you don't know the answer, just say that you don't know.

Context: {context}"),
    ("user", "{question}")
])

# 4. Construct the RAG Chain using LCEL
# This chain defines the flow: retrieve context, format prompt, invoke LLM, parse output.
rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | rag_prompt_template
    | llm
    | StrOutputParser()
)

print("RAG chain constructed.")
print("Setup complete. You can now proceed to implement the FastAPI application.")


### Your Turn: Implement the FastAPI Backend

Now it's your turn to implement the FastAPI application. Below, you'll find a code cell where you should write your solution. Remember to:

1.  Import necessary FastAPI and Pydantic components.
2.  Define `QueryRequest` and `QueryResponse` Pydantic models.
3.  Create a FastAPI app instance.
4.  Implement the `/query` POST endpoint, integrating the `rag_chain` defined in the setup cell.
5.  Include basic error handling.

Once implemented, you can run your FastAPI application using `uvicorn` from your terminal (or a new terminal tab if running in a Jupyter environment):

```bash
uvicorn main:app --reload --port 8000
```

(Assuming your Python file is named `main.py` and your FastAPI app instance is named `app`).

Then, you can test it using `curl` or a tool like Postman/Insomnia, or by navigating to `http://localhost:8000/docs` for the interactive OpenAPI documentation.

```bash
curl -X POST "http://localhost:8000/query" \
     -H "Content-Type: application/json" \
     -d '{"query": "What is the capital of France?"}'
```


In [ ]:
import os
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from typing import Dict, Any

# Assuming rag_chain is already defined in the setup cell above
# from langchain_core.runnables import Runnable
# from langchain_community.vectorstores import FAISS
# from langchain_openai import OpenAIEmbeddings, ChatOpenAI
# from langchain_core.prompts import ChatPromptTemplate
# from langchain_core.output_parsers import StrOutputParser
# from langchain_core.runnables import RunnablePassthrough

# --- Pydantic Models for Request and Response ---
class QueryRequest(BaseModel):
    """Request model for a RAG query."""
    query: str

class QueryResponse(BaseModel):
    """Response model for a RAG query result."""
    answer: str
    source_documents: List[str] = [] # Optional: to show retrieved context

# --- FastAPI Application Instance ---
app = FastAPI(
    title="RAG FastAPI Backend",
    description="A REST API for a Retrieval-Augmented Generation (RAG) system using LangChain and FastAPI.",
    version="1.0.0"
)

# --- API Endpoint ---
@app.post("/query", response_model=QueryResponse)
async def query_rag_system(request: QueryRequest):
    """
    Processes a user query through the RAG system and returns an answer.
    """
    try:
        # Invoke the RAG chain asynchronously
        # Note: LangChain's .invoke() is synchronous. For true async, use .ainvoke()
        # if all components in the chain support it. For this exercise, .invoke()
        # within an async endpoint is common practice.
        
        # To get source documents, we need to modify the chain slightly or invoke retriever separately.
        # For simplicity, let's just get the answer first.
        answer = rag_chain.invoke(request.query)

        # To include source documents, we'd typically modify the rag_chain to return them
        # or call the retriever separately. For this solution, let's simulate it
        # by directly calling the retriever to show what would be retrieved.
        retrieved_docs = retriever.invoke(request.query)
        source_texts = [doc.page_content for doc in retrieved_docs]

        return QueryResponse(answer=answer, source_documents=source_texts)
    except Exception as e:
        # Log the error for debugging purposes (in a real app, use a proper logger)
        print(f"Error processing query: {e}")
        raise HTTPException(status_code=500, detail=f"Internal Server Error: {e}")

# --- Health Check Endpoint (Optional but Recommended) ---
@app.get("/health")
async def health_check():
    """
    Health check endpoint to verify the API is running.
    """
    return {"status": "ok", "message": "RAG API is up and running!"}

# To run this application, save the code above into a file (e.g., `main.py`)
# and execute `uvicorn main:app --reload --port 8000` in your terminal.

# Example `requirements.txt` content for this application:
# fastapi
# uvicorn[standard]
# langchain
# langchain-openai
# pydantic
# faiss-cpu
# python-dotenv
